###  S3 Pre-Signed URLs to Static URLs Script

**Step 1:** Configure SSO for prod bucket access for downloading prod doubt images locally using `aws configure sso` and then once it is setup, use `aws sso login`

In [ ]:
#!aws sso login --profile data-science-prod-access-596691011161

In [1]:
# --- Cell 1: Setup and Config ---

import os
import re
import pandas as pd
import boto3
from urllib.parse import urlparse
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ========== CONFIG ==========

# AWS
# AWS_PROFILE = "your-sso-profile-name"   # replace with your SSO profile name
AWS_REGION = "ap-south-1"               # region of your bucket
BUCKET_NAME = "doubt-qc"                # destination bucket

# Folders / Files
DOWNLOAD_DIR = "/Users/simrannaik/Desktop/bots/svg/svg_data_800_PCMB/images"
UPLOAD_PREFIX = "PCMB_500_svg_PCMB"   # folder name inside S3 ## folder in s3 should be same to the local folder here 

# RAW_CSV = "/Users/jateenleo/Documents/bot_agent_run/spa_acc_testing_data/spa_acc_all.csv"
# INTERMEDIATE_1 = "/Users/jateenleo/Documents/bot_agent_run/spa_acc_testing_data/spa_acc_all.csv"
# CLEANED_CSV = "/Users/jateenleo/Documents/bot_agent_run/spa_acc_testing_data/spa_acc_all.csv"
# WITH_LOCAL_PATHS = "/Users/jateenleo/Documents/bot_agent_run/spa_acc_testing_data/spa_acc_all.csv"
# FINAL_CSV = "/Users/jateenleo/Documents/bot_agent_run/spa_acc_testing_data/spa_acc_all.csv"

RAW_CSV = INTERMEDIATE_1 = CLEANED_CSV = WITH_LOCAL_PATHS = FINAL_CSV = "/Users/simrannaik/Desktop/bots/svg/svg_data_800_PCMB/PCMB_500_svg_PCMB.csv"
session = boto3.Session()
s3 = session.client("s3")


**Step 2:** Code to extract image_key from current_input_image_url

In [2]:
# --- Cell 2: Extract image_key from current_input_image_url ---

def extract_image_key(url: str) -> str:
    """Extract file name like AD-2025-09-30-23-41-33-418.jpeg from URL."""
    if not isinstance(url, str) or not url.strip():
        return ""
    match = re.search(r"/([^/?#]+\.[A-Za-z0-9]{3,4})(?:\?|$)", url)
    return match.group(1) if match else ""

df = pd.read_csv(RAW_CSV)
df["image_key"] = df["current_input_image_url"].apply(extract_image_key)

df.to_csv(INTERMEDIATE_1, index=False)
print(f"✅ Extracted image_key and saved to: {INTERMEDIATE_1}")
df.head(3)



✅ Extracted image_key and saved to: /Users/simrannaik/Desktop/bots/svg/svg_data_800_PCMB/PCMB_500_svg_PCMB.csv


,request_timestamp,classification_type,academic_subtype,stream,student_class,subject,topics,current_input_text,current_input_image_url,current_input_image_ocr,...,bot_response_model_used,source,test_practice_context,is_diagram_present,qa_complete_context,theory_complete_context,subject_normalized,diagram_required,justification,image_key
0,2026-06-17 10:46:55,ACADEMIC,STUDY_RELATED_PROBLEM_SOLVING,STREAM_PRE_MEDICAL,Class 12 Plus,Biology,"[""Structural Organisation In Animals""]",NaN,https://doubt-qc.s3.amazonaws.com/question_thu...,"""\nWhich part gets reduced in the alimentary c...",...,exact_match_direct_llm_solution_used,TEST_AND_ASSESSMENT,"{""TEST_CONTEXT"":""{\""test_id\"": \""test_H530QrXg...",False,<Context1>: <Question>: Which part get reduced...,<Context1>:DIGESTIVE SYSTEM Function of digest...,Biology,No,The question and solution involve a conceptual...,2cbb3417-7a3e-11ef-8048-ba2447f841c3.jpg
1,2026-06-11 13:13:03,ACADEMIC,STUDY_RELATED_PROBLEM_SOLVING,STREAM_PRE_MEDICAL,Class 12 Plus,Biology,"[""Human Health And Disease""]",NaN,https://doubt-qc.s3.amazonaws.com/question_thu...,"""Identify the molecules (a) and (b) shown belo...",...,exact_match_direct_llm_solution_used,TEST_AND_ASSESSMENT,"{""TEST_CONTEXT"":""{\""test_id\"": \""test_JjYcAHqi...",False,<Context1>: <Question>: Identify the molecules...,<Context1>:COCA ALKALOID OR COCAINE These are ...,Biology,No,The INPUT_IMAGE already contains the clear che...,a0924985-51d8-11f1-9bc2-961d34c7f4b0.jpg
2,2026-06-13 02:56:55,ACADEMIC,STUDY_RELATED_PROBLEM_SOLVING,STREAM_PRE_MEDICAL,Class 12,Biology,"[""Sexual Reproduction In Flowering Plants""]",NaN,https://doubt-qc.s3.amazonaws.com/question_thu...,II\nWhich one is essential for activation of p...,...,neet_biobot_problem_solving_solution_gen,TEST_AND_ASSESSMENT,"{""TEST_CONTEXT"":""{\""test_id\"": \""test_XL1jkBJS...",False,<Context1>: <Question>: Which of the following...,<Context1>:Essential elements that activate or...,Biology,No,The question is a factual recall of a biochemi...,8df0928b-fd98-11ef-a32d-36648b52a590.jpg


**Step 3**: Code to drop rows with URL but missing image_key

In [3]:
# --- Cell 3: Drop rows with URL but missing image_key ---

mask_url_present = df["current_input_image_url"].notna() & df["current_input_image_url"].astype(str).str.strip().ne("")
mask_key_missing = df["image_key"].astype(str).str.strip().eq("")

problem_rows = df[mask_url_present & mask_key_missing]
print(f"⚠️ Found {len(problem_rows)} problematic rows where URL exists but no image_key")

# Drop them
df_clean = df.drop(problem_rows.index).reset_index(drop=True)
df_clean.to_csv(CLEANED_CSV, index=False)

print(f"✅ Cleaned dataset saved: {CLEANED_CSV}", df_clean.shape)


⚠️ Found 0 problematic rows where URL exists but no image_key
✅ Cleaned dataset saved: /Users/simrannaik/Desktop/bots/svg/svg_data_800_PCMB/PCMB_500_svg_PCMB.csv (500, 21)


Make sure to create the session again so that these new credentials with proper access are used to do the upload

**Step 4:** Code to deterministically download from S3 by positional mapping

In [4]:
# df_clean['image_local_path'] = df_clean['image_key'].apply(
#     lambda x: os.path.join("/Users/jateenleo/Downloads/downloads", x) if pd.notna(x) and isinstance(x, str) and x.strip() else ""
# )

## this is to send url images to allen digital produciton = data science prod access

In [5]:
# --- Cell 4 (Fixed): Deterministic download from S3 by positional mapping ---

from concurrent.futures import ThreadPoolExecutor, as_completed
from botocore.exceptions import ClientError
import urllib.parse, os

PROFILE = "data-science-prod-access-596691011161"
AWS_REGION = "ap-south-1"

# One-time CLI login for this profile
!aws sso login --profile {PROFILE}

# Ensure boto3 uses the same SSO profile/region
os.environ["AWS_PROFILE"] = PROFILE  # optional but helpful
session = boto3.Session(profile_name=PROFILE, region_name=AWS_REGION)
s3 = session.client("s3")
sts = session.client("sts")

# DOWNLOAD_DIR = "/Users/om.singh/Desktop/ds_helper/NEET_ChemBot_images"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
SOURCE_BUCKET = "ap-south-1-prod-doubts"

def download_from_s3_by_key(image_key):
    """
    Downloads an image by key (decoded) from S3 -> local directory.
    Returns local absolute path if successful, else "".
    """
    if not isinstance(image_key, str) or not image_key.strip():
        return ""
    
    # Decode any URL-encoded filenames like "2025-09-13%2015%3A25%3A41.jpeg"
    decoded_key = urllib.parse.unquote(image_key.strip())
    
    # Build local path
    local_filename = os.path.basename(decoded_key)
    local_path = os.path.join(DOWNLOAD_DIR, local_filename)
    
    # Try downloading
    try:
        s3.download_file(SOURCE_BUCKET, decoded_key, local_path)
        return os.path.abspath(local_path)
    except ClientError as e:
        print(f"⚠️ Failed to download {decoded_key}: {e.response['Error']['Code']}")
        return ""

# --- Deterministic parallel download ---
local_paths = [""] * len(df_clean)

with ThreadPoolExecutor(max_workers=8) as executor:
    future_to_pos = {
        executor.submit(download_from_s3_by_key, row.image_key): pos
        for pos, row in enumerate(df_clean.itertuples(index=False))
    }
    for future in tqdm(as_completed(future_to_pos), total=len(future_to_pos), desc="Downloading from S3"):
        pos = future_to_pos[future]
        local_paths[pos] = future.result()

df_clean["image_local_path"] = local_paths

downloaded_count = sum(bool(p) for p in local_paths)
print(f"✅ Downloaded {downloaded_count} / {len(df_clean)} images successfully.")
print("📁 Added 'image_local_path' column (local filesystem paths).")


Attempting to automatically open the SSO authorization page in your default browser.
If the browser does not open or you wish to use a different device to authorize this request, open the following URL:

https://d-9f6767f353.awsapps.com/start/#/device

Then enter the code:

KZPR-CPNX
Successfully logged into Start URL: https://identitycenter.amazonaws.com/ssoins-6595771594cdcd0c


⚠️ Failed to download c2622644-99c6-11f0-b69e-feabf7a5b792.jpg: 404
⚠️ Failed to download 1e1ba385-aff0-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 2cbb3417-7a3e-11ef-8048-ba2447f841c3.jpg: 404
⚠️ Failed to download 03aae198-51e7-11f1-818f-de7031dccd34.jpg: 404
⚠️ Failed to download a0924985-51d8-11f1-9bc2-961d34c7f4b0.jpg: 404
⚠️ Failed to download 92c32e5f-fbdc-11ef-983f-1a368dbeb272.jpg: 404
⚠️ Failed to download 7ea1c55f-50e2-11f1-bbb1-6a8a5bffcb8c.jpg: 404
⚠️ Failed to download 8df0928b-fd98-11ef-a32d-36648b52a590.jpg: 404
⚠️ Failed to download e8e97dc0-542b-11f1-9ca0-5eb15f665d00.jpg: 404
⚠️ Failed to download aa6ff983-bd1d-11ef-b18b-2e9041356e1e.jpg: 404
⚠️ Failed to download e0092a85-1e98-11f1-8403-3a1b3a0f4330.jpg: 404
⚠️ Failed to download 31f083bf-5501-11f1-b7ea-92593d714e43.jpg: 404
⚠️ Failed to download 67b0cef5-aff3-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 421fdcad-afdf-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download b4680491-0f18-11f1-9904-42

⚠️ Failed to download 27d90158-664a-11f1-a618-22c014715d48.jpg: 404
⚠️ Failed to download cbff7442-afdf-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 6a9c8ee2-b009-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 1df99533-affa-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 4a7aceca-affc-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download fded0709-aff6-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 9973499b-51ab-11f1-9bc2-961d34c7f4b0.jpg: 404
⚠️ Failed to download 749b1b9b-6189-11f1-83ac-72a1091f75c8.jpg: 404
⚠️ Failed to download 641e99a1-afe1-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 704f5d00-aff3-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 3bf7b189-d85b-11ee-bde4-02f11adfad73.jpg: 404
⚠️ Failed to download 703b40e6-f1b1-11ee-b865-0e67afbcd4cf.jpg: 404
⚠️ Failed to download 2e514f49-affe-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 931aca97-ea9a-11ef-be91-9acecaa40632.jpg: 404
⚠️ Failed to download 400d1454-582a-11f1-9911-36

⚠️ Failed to download 720c4ee5-aff3-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download f460fd2d-2ef1-11ef-9e0c-c6c790f86a40.jpg: 404
⚠️ Failed to download 64740a69-aff3-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download bd0b5fa5-1218-11f0-9978-46eb17aad353.jpg: 404
⚠️ Failed to download 43f53692-11d5-11f0-bc0f-cad31fffd7fd.jpg: 404
⚠️ Failed to download 92c32e5f-fbdc-11ef-983f-1a368dbeb272.jpg: 404
⚠️ Failed to download d9c161fd-14fd-11f0-9316-ba19cabd737a.jpg: 404
⚠️ Failed to download 68b81d95-affd-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download b24e60e7-51d3-11f1-a74d-2a2215ff482a.jpg: 404
⚠️ Failed to download 44c568d1-afd7-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 31f083bf-5501-11f1-b7ea-92593d714e43.jpg: 404
⚠️ Failed to download 124fdca4-58ec-11f1-9bb5-963c28b44798.jpg: 404
⚠️ Failed to download 65b1c9f9-542b-11f1-abf6-1a301deff022.jpg: 404
⚠️ Failed to download 87797b1c-61fd-11f0-972b-26518c31d9af.jpg: 404
⚠️ Failed to download 6e2d4848-affc-11ee-8d4d-86

⚠️ Failed to download 06b73129-6ca7-11f1-9247-e6737d9be42e.jpg: 404
⚠️ Failed to download 6b4a9bb4-affd-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download cb3f97c3-136a-11f0-84c9-5a3f5d54888c.jpg: 404
⚠️ Failed to download d758c5a0-fe37-11ef-86a3-628933a7c910.jpg: 404
⚠️ Failed to download MD12BOPM06SCM-0064_1.png: 404
⚠️ Failed to download 4210a3b5-afd9-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 82da36ec-aff3-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 13af95e2-5343-11f1-9e0b-da1114a7b935.jpg: 404
⚠️ Failed to download d2ddea25-4e5d-11ef-be52-56f8d447ea92.jpg: 404
⚠️ Failed to download 1688c5b1-1502-11f0-9316-ba19cabd737a.jpg: 404
⚠️ Failed to download f3c398c7-5f45-11f1-b458-02c1a7a5ab07.jpg: 404
⚠️ Failed to download c2dc6e2a-afd9-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download b97ba2f2-6c87-11f1-a429-f6e2eaa422f6.jpg: 404
⚠️ Failed to download b24becb7-5186-11ef-be52-56f8d447ea92.jpg: 404
⚠️ Failed to download fb2bedc0-afe3-11ee-8d4d-86c3d2487f1d.jpg: 

...

⚠️ Failed to download ab5f2804-affe-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download c24f140f-33d8-11f0-afaa-9294017e247d.jpg: 404
⚠️ Failed to download 52a36cad-b009-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 3570cbb1-afff-11ee-8d4d-86c3d2487f1d.jpg: 404


⚠️ Failed to download a55346cc-51bf-11f1-acbf-82ebe5888d25.jpg: 404
⚠️ Failed to download 9cbe4146-afd6-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 4e9b4eb9-afd6-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 94c2a7ec-afd3-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 068135fe-afd8-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 63e51442-5b21-11f1-861c-feab47bad63d.jpg: 404
⚠️ Failed to download b933d395-affc-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download fe58861f-7f26-11ef-a183-7e85891d5db7.jpg: 404
⚠️ Failed to download 13b16f27-aff2-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 34bd40c8-afe1-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 750ac015-afd8-11ee-8d4d-86c3d2487f1d.jpg: 404


⚠️ Failed to download e86ac96a-afec-11ee-8d4d-86c3d2487f1d.jpg: 404
⚠️ Failed to download 9ed8d74a-06e1-11ef-9e4f-824a7cc4f1f1.jpg: 404
⚠️ Failed to download c0b1943b-5126-11f1-a74d-2a2215ff482a.jpg: 404


⚠️ Failed to download 4d03365a-5410-11f1-864c-8a1e6f5281a1.jpg: 404
⚠️ Failed to download 640925d4-1615-11f1-aaa8-a286e742473e.jpg: 404
⚠️ Failed to download 6fe3b82d-07d8-11f1-9722-bac67150b975.jpg: 404


✅ Downloaded 248 / 500 images successfully.
📁 Added 'image_local_path' column (local filesystem paths).


## this is to send url images to data science = power user access 

At this point, you've downloaded the prod doubt images locally on the system. Now, you need to upload them to the datascience s3 bucket in a folder of your choice, post which the file will be accessible to you as a **static URL**

For this, you need to configure the SSO again using `aws configure sso` but this time for the **datascience profile access**, and then do `aws sso login`

**Step 5:** Code to decode image_key before upload and upload to s3 bucket for static image url

In [6]:
import os
from urllib.parse import unquote
from concurrent.futures import ThreadPoolExecutor, as_completed
from botocore.exceptions import ClientError, TokenRetrievalError
import boto3

PROFILE = "PowerUserAccess-840443303776"
AWS_REGION = "ap-south-1"

# One-time CLI login for this profile
!aws sso login --profile {PROFILE}

# Ensure boto3 uses the same SSO profile/region
os.environ["AWS_PROFILE"] = PROFILE  # optional but helpful
session = boto3.Session(profile_name=PROFILE, region_name=AWS_REGION)
s3 = session.client("s3")
sts = session.client("sts")

# Preflight: ensure credentials are valid before spawning threads
sts.get_caller_identity()

SOURCE_BUCKET = "ap-south-1-prod-doubts"
DEST_BUCKET = "doubt-qc"
# UPLOAD_PREFIX = "input_doubts_analysis"  # set to your desired prefix

def upload_to_s3_and_get_url(local_path, image_key):
    if not isinstance(local_path, str) or not local_path.strip() or not os.path.exists(local_path):
        return ""
    decoded_key = unquote(image_key.strip())
    s3_key = f"{UPLOAD_PREFIX}/{decoded_key}"
    try:
        s3.upload_file(local_path, DEST_BUCKET, s3_key)
        return f"https://{DEST_BUCKET}.s3.{AWS_REGION}.amazonaws.com/{s3_key}"
    except (ClientError, TokenRetrievalError) as e:
        print(f"⚠️ Failed to upload {decoded_key}: {e}")
        return ""

# Thread-safe, order-preserving upload
url_list = [""] * len(df_clean)
with ThreadPoolExecutor(max_workers=8) as executor:
    future_to_pos = {
        executor.submit(upload_to_s3_and_get_url, row.image_local_path, row.image_key): pos
        for pos, row in enumerate(df_clean.itertuples(index=False))
    }
    for future in tqdm(as_completed(future_to_pos), total=len(future_to_pos), desc="Uploading (decoded keys)"):
        pos = future_to_pos[future]
        url_list[pos] = future.result()

df_clean["static_image_url"] = url_list
uploaded_count = sum(bool(u) for u in url_list)
print(f"✅ Uploaded {uploaded_count} / {len(df_clean)} images successfully (decoded keys).")
print("📄 Added static_image_url column (S3 HTTPS URLs, human-readable, no presign).")

Attempting to automatically open the SSO authorization page in your default browser.
If the browser does not open or you wish to use a different device to authorize this request, open the following URL:

https://d-9f6767f353.awsapps.com/start/#/device

Then enter the code:

HXLV-FSKS
Successfully logged into Start URL: https://identitycenter.amazonaws.com/ssoins-6595771594cdcd0c


Uploading (decoded keys): 100%|██████████| 500/500 [00:03<00:00, 155.72it/s]

✅ Uploaded 248 / 500 images successfully (decoded keys).
📄 Added static_image_url column (S3 HTTPS URLs, human-readable, no presign).


**Step 6:** Fallback logic for missing local paths - this is just some post-check cleanup code

In [7]:
# --- Cell 6: Fallback logic for missing local paths ---

mask_url_present = (
    df_clean["current_input_image_url"].notna() &
    df_clean["current_input_image_url"].astype(str).str.strip().ne("")
)

mask_local_missing = (
    df_clean["image_local_path"].isna() |
    df_clean["image_local_path"].astype(str).str.strip().eq("")
)

mask_static_missing = (
    df_clean["static_image_url"].isna() |
    df_clean["static_image_url"].astype(str).str.strip().eq("")
)

fallback_mask = mask_url_present & mask_local_missing & mask_static_missing
num_fallback = fallback_mask.sum()

print(f"⚙️ Found {num_fallback} rows needing fallback")

df_clean.loc[fallback_mask, "static_image_url"] = df_clean.loc[fallback_mask, "current_input_image_url"]

print("✅ Applied fallback logic successfully.")


⚙️ Found 195 rows needing fallback
✅ Applied fallback logic successfully.


**Step 7:** Save final CSV 

In [8]:
# --- Cell 7: Save final CSV and run QA summary ---

df_clean.to_csv(FINAL_CSV, index=False)
print(f"✅ Final dataset saved: {FINAL_CSV}")

# --- QA summary ---
total_rows = len(df_clean)
filled_static = df_clean["static_image_url"].astype(str).str.strip().ne("").sum()
fallback_count = df_clean.loc[
    (df_clean["image_local_path"].astype(str).str.strip().eq("")) &
    (df_clean["static_image_url"].astype(str).str.strip().ne(""))
].shape[0]

print(f"""
📊 QA Summary:
---------------------------------
Total Rows: {total_rows}
Rows with static_image_url: {filled_static}
Fallback Rows: {fallback_count}
---------------------------------
""")

df_clean.sample(min(5, len(df_clean)))


✅ Final dataset saved: /Users/simrannaik/Desktop/bots/svg/svg_data_800_PCMB/PCMB_500_svg_PCMB.csv

📊 QA Summary:
---------------------------------
Total Rows: 500
Rows with static_image_url: 443
Fallback Rows: 195
---------------------------------



,request_timestamp,classification_type,academic_subtype,stream,student_class,subject,topics,current_input_text,current_input_image_url,current_input_image_ocr,...,test_practice_context,is_diagram_present,qa_complete_context,theory_complete_context,subject_normalized,diagram_required,justification,image_key,image_local_path,static_image_url
191,2026-06-19 07:42:18,ACADEMIC,STUDY_RELATED_PROBLEM_SOLVING,STREAM_PRE_MEDICAL,Class 12,Chemistry,"[""Chemical Equilibrium""]",NaN,https://ap-south-1-prod-doubts.s3.ap-south-1.a...,52. One mole of \( \mathrm{N}_{2} \mathrm{O}_{...,...,"{""SOURCE"":""DOUBTS"",""REFERENCE_SOLUTION"":""""}",False,<Context1>: <Question>: One mole of \( \mathrm...,<Context1>:Chunk 11:\nThe following data were ...,Chemistry,No,The problem is a standard stoichiometry and id...,AD-2026-06-19-13-12-14-040.jpeg,/Users/simrannaik/Desktop/bots/svg/svg_data_80...,https://doubt-qc.s3.ap-south-1.amazonaws.com/P...
449,2026-06-25 13:57:47,ACADEMIC,STUDY_RELATED_PROBLEM_SOLVING,STREAM_PRE_MEDICAL,Class 11,Physics,"[""Vectors""]",NaN,https://doubt-qc.s3.amazonaws.com/question_thu...,"""\nThe value of \( (\vec{A}+\vec{B}) \times(\v...",...,"{""TEST_CONTEXT"":""{\""test_id\"": \""ct_yeFZBQgX0S...",False,<Context1>: <Question>: The value of\n\[\n(\ve...,<Context1>:Find \(\vec{A} \cdot \vec{B}\). if ...,Physics,No,The problem is a purely symbolic algebraic man...,9cbe4146-afd6-11ee-8d4d-86c3d2487f1d.jpg,,https://doubt-qc.s3.amazonaws.com/question_thu...
32,2026-06-15 07:45:51,ACADEMIC,STUDY_RELATED_PROBLEM_SOLVING,STREAM_PRE_MEDICAL,Class 12 Plus,Biology,"[""Cell Cycle And Cell Division""]",NaN,https://doubt-qc.s3.amazonaws.com/question_thu...,"""\nWhich is not true for the nucleus of prokar...",...,"{""TEST_CONTEXT"":""{\""test_id\"": \""test_5GQiRJrW...",False,<Context1>: <Question>: Which is not true for ...,"<Context1>:Some eubacteria, such as purple bac...",Biology,No,The question and solution are based on factual...,4680a189-5cbd-11f1-891a-52dd379769ff.jpg,,https://doubt-qc.s3.amazonaws.com/question_thu...
327,2026-06-16 14:23:48,ACADEMIC,STUDY_RELATED_PROBLEM_SOLVING,STREAM_JEE_MAINS,Class 12,Maths,"[""Circle""]",NaN,https://ap-south-1-prod-doubts.s3.ap-south-1.a...,"a) \( (1,1) \)\nb) \( (1.2) \)\n\nPart-II (Num...",...,"{""SOURCE"":""DOUBTS"",""REFERENCE_SOLUTION"":""""}",False,<Context1>: <Question>: The circles x^{2} + y^...,NaN,Mathematics,Yes,The problems involve complex spatial relations...,AD-2026-06-16-19-53-02-175.jpeg,/Users/simrannaik/Desktop/bots/svg/svg_data_80...,https://doubt-qc.s3.ap-south-1.amazonaws.com/P...
116,2026-06-10 15:36:26,ACADEMIC,STUDY_RELATED_PROBLEM_SOLVING,STREAM_PRE_MEDICAL,Class 12,Biology,"[""Biomolecules""]",NaN,https://doubt-qc.s3.amazonaws.com/question_thu...,"""\nWhat is the state of heart in the interval ...",...,"{""TEST_CONTEXT"":""{\""test_id\"": \""test_IS2taZVY...",False,<Context1>: <Question>: What is the state of h...,<Context1>:(Joint Diastole 0.8-0.4=0.4 sec. Pe...,Biology,Yes,A cardiac cycle diagram or timeline is essenti...,4210a3b5-afd9-11ee-8d4d-86c3d2487f1d.jpg,,https://doubt-qc.s3.amazonaws.com/question_thu...


In [9]:
print(df_clean.shape)
df_clean = df_clean[~df_clean['static_image_url'].str.startswith('https://ap-south-1-prod-doubts', na=False)]
print(df_clean.shape)

(120, 46)
(120, 46)


In [10]:
df_clean.to_csv(FINAL_CSV, index=False)

In [11]:
import pandas as pd 
FINAL_CSV = "/Users/simrannaik/Desktop/bots/neet_bots/neet_model_change_june_08_14_biology_first_response.csv"
data = pd.read_csv(FINAL_CSV)

In [12]:
data.subject.value_counts()

subject
Biology    120
Name: count, dtype: int64

In [13]:
import pandas as pd 
FINAL_CSV = "/Users/simrannaik/Desktop/bots/neet_bots/bot-verifier-script/neet_model_change_data_math.csv"
data = pd.read_csv(FINAL_CSV)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/simrannaik/Desktop/bots/neet_bots/bot-verifier-script/neet_model_change_data_math.csv'

In [49]:
data.subject.value_counts()

subject
Biology      120
Chemistry    120
Physics      120
Maths        120
Name: count, dtype: int64

In [13]:
import pandas as pd

# Load CSV
CSV_PATH = "/Users/simrannaik/Desktop/bots/neet_bots/jee_model_change_may_01_31_maths_first_response_latency_check_60.csv"

df = pd.read_csv(CSV_PATH)

# Replace with your latency column name if different
LATENCY_COL = "latency_seconds"

# Optional: remove rows with missing latency
df = df[df[LATENCY_COL].notna()]

# Optional: convert to numeric
df[LATENCY_COL] = pd.to_numeric(df[LATENCY_COL], errors="coerce")
df = df[df[LATENCY_COL].notna()]

# Calculate subject-wise metrics
subject_latency = (
    df.groupby("subject")[LATENCY_COL]
      .agg(
          count="count",
          avg_latency="mean",
          p95_latency=lambda x: x.quantile(0.95),   # P95 (s)
          p99_latency=lambda x: x.quantile(0.99),
          median_latency="median",
          min_latency="min",
          max_latency="max",
      )
      .reset_index()
)

# Round values
latency_cols = [
    "avg_latency",
    "p95_latency",
    "p99_latency",
    "median_latency",
    "min_latency",
    "max_latency",
]

subject_latency[latency_cols] = subject_latency[latency_cols].round(3)

# Sort by average latency (optional)
subject_latency_df = subject_latency.sort_values(
    "avg_latency", ascending=False
)
subject_latency_df

,subject,count,avg_latency,p95_latency,p99_latency,median_latency,min_latency,max_latency
0,Maths,60,8.155,18.878,33.152,5.492,2.282,40.191
